This script matches spots from 2 different channels to their nearest neighbour. 1 spot is assigned only to 1 pair!

## Input:
`.csv` file of spots in each of the 2 channels, containg following columns:
- *img* - image name
- *channel* - channel number the spot belongs to  
- *x, y* and *z* coordiantes

## Output
- `distances.csv` with information on the image name, x,y,z positions of matched spots and the distances between them \[nm\].

# Functions and imports
*also part of `pipelines/fish_utils`*

In [14]:
import pandas as pd
import numpy as np
from collections import defaultdict
from itertools import product
from scipy.optimize import linear_sum_assignment

In [12]:
def detect_spot_pairs(path, out, ch, voxel_size=(300, 130, 130)):
    df = pd.read_csv(path)
    df['img'] = df['img'].apply(lambda x: x.rsplit('_', 1)[0])
    
    result = defaultdict(list)
    voxel_size = np.array(voxel_size)
    
    # Group the DataFrame by 'img' 
    grouped = df.groupby(['img'])

    for (img), group_df in grouped:
        spot_coords_ch1 = group_df.loc[group_df['channel'] == ch[0], ['x', 'y', 'z']].values
        spot_coords_ch2 = group_df.loc[group_df['channel'] == ch[1], ['x', 'y', 'z']].values
        distances = np.zeros((len(spot_coords_ch1), len(spot_coords_ch2)))

        for (i1,c1), (i2, c2) in product(enumerate(spot_coords_ch1), enumerate(spot_coords_ch2)):
            distances[i1, i2] = np.linalg.norm((c1 - c2)*voxel_size) # np.sqrt(np.sum((c1 - c2)**2))

        row_ind, col_ind = linear_sum_assignment(distances)
        
        for ri, ci in zip(row_ind, col_ind):
            result['img'].append(img)
            result['distance_nm'].append(distances[ri,ci])
            
            for dim_i, dim in enumerate('zyx'):
                result[f'pos_{dim}_ch1'].append(spot_coords_ch1[ri][dim_i])
                result[f'pos_{dim}_ch2'].append(spot_coords_ch2[ci][dim_i])

    result_df = pd.DataFrame(result)
    result_df.to_csv(out, index=False)

# Match spots and calculate distances

In [15]:
path = "/data/.../" #upper level experiment folder
path_spots = f"{path}/detections/merge.csv"
out_distances = f"{path}/distances.csv"
channels = [1,2] # which channels to match
voxel_size=(300, 130, 130) #sizes of zyx [nm]

detect_spot_pairs(path_spots,out_distances,channels,voxel_size)